In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

import matplotlib.pyplot as plt

from tqdm import tqdm

from dataset_classes.lazy_dataset import LazyDataset
from src.config_utils import setup_config_variables

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

In [ ]:
config = setup_config_variables()

NORMALISE_OUTPUT = False

In [ ]:
inv_apsdvs_max = 1 / 255

# These normalisation functions are not standardised intentionally, to take advantage of pre-computing with some known values.
# used to normalise the aps and dvs frames to the range [0, 1]
def normalise_dvsaps(data):
    # shorthand for (data - self.apsdvs_min) / (self.apsdvs_max - self.apsdvs_min) since we know the min is 0
    new_data = np.empty_like(data, dtype=config["DTYPE"])
    new_data = data * inv_apsdvs_max
    return new_data

transform = transforms.Compose([
    lambda x: normalise_dvsaps(x),
    transforms.ToPILImage(),
    transforms.ToTensor(),
])

# normalise the steering values from -180 to 180 to -1 to 1
# Precompute normalization coefficients
steering_slope = 1 / 180  # (output_max - output_min)/(input_max - input_min) = (1-(-1))/(180-(-180))
def normalise_steering(data):
    return data * steering_slope

# used for getting the data out at the end.
inv_steering_slope = 180
def denormalise_steering(data):
    return data * inv_steering_slope

if NORMALISE_OUTPUT:
    target_transform = transforms.Compose([
        lambda x: normalise_steering(x)
    ])
else:
   target_transform = transforms.Compose([]) 

train_dataset = LazyDataset(config["TRAINING_DIR"], transform=transform, target_transform=target_transform)
test_dataset = LazyDataset(config["TEST_DIR"], transform=transform, target_transform=target_transform)

# load the data, can implement batching etc. here
train_loader = DataLoader(train_dataset, batch_size=config["BATCH_SIZE"], shuffle=False, drop_last=True, num_workers=4, pin_memory=True, prefetch_factor=5)
test_loader = DataLoader(test_dataset, batch_size=config["BATCH_SIZE"], shuffle=False, drop_last=True, num_workers=4, pin_memory=True, prefetch_factor=5)

In [ ]:
train_angles = []
# Initialize variables to store running stats
channel_sum = torch.zeros(2)
channel_sum_sq = torch.zeros(2)
pixel_count = 0

for i, (combined_images, steering_angle) in tqdm(enumerate(train_loader), total=len(train_loader)):
    # speed up
    if i % 100 != 0:
        continue
    b, t, c, h, w = combined_images.shape
    images = combined_images.permute(2, 0, 1, 3, 4)
    # Sum across all dimensions except channel dimension
    channel_sum += images.sum(dim=(1, 2, 3, 4))
    channel_sum_sq += (images ** 2).sum(dim=(1, 2, 3, 4))
    # Count number of values per channel
    pixel_count += b * t * h * w

    steering_angle = steering_angle.squeeze(1)
    results = steering_angle.numpy()
    if NORMALISE_OUTPUT:
        results = denormalise_steering(results)
    train_angles.extend(results)

# Calculate mean and std
channel_mean = channel_sum / pixel_count
channel_std = torch.sqrt(channel_sum_sq / pixel_count - channel_mean ** 2)

print(f"Training\nAPS (channel 0) - Mean: {channel_mean[0]:.4f}, StdDev: {channel_std[0]:.4f}")
print(f"DVS (channel 1) - Mean: {channel_mean[1]:.4f}, StdDev: {channel_std[1]:.4f}")

test_angles = []
for i, (combined_images, steering_angle) in tqdm(enumerate(test_loader), total=len(test_loader)):
    # speed up
    if i % 100 != 0:
        continue
    steering_angle = steering_angle.squeeze(1)
    results = steering_angle.numpy()
    if NORMALISE_OUTPUT:
        results = denormalise_steering(results)
    test_angles.extend(results)

print(f"Test angles have a mean of {np.mean(test_angles)}, with a stddev of {np.std(test_angles)}")

fig = plt.figure(figsize=(20, 12))
plt.subplot(2, 1, 1)
plt.hist(train_angles, bins=360, alpha=0.7)
plt.title('Distribution of Steering Angles, Training Set')
plt.xlabel('Steering Angle')
plt.ylabel('Frequency')


plt.subplot(2, 1, 2)
plt.hist(test_angles, bins=360, alpha=0.7)
plt.title('Distribution of Steering Angles, Test Set')
plt.xlabel('Steering Angle')
plt.ylabel('Frequency')

plt.show()